# AI 4200 Project 1: Residual MLPs on MNIST, CIFAR-10, and ImageNet 32×32

This notebook is the **test harness** for Project 1. It does three things:

1. Installs your package from GitHub.
2. Downloads the three datasets. It saves a copy in `data/`, so later runs load from disk instead of downloading again.
3. Calls your `train()` and `predict()` on each task and reports test accuracy.

Develop your model in your own environment. Use this notebook to check that your submission runs end to end, exactly as it will when it is graded.

## The tasks

| Task | Images | Classes | Train | Test |
|---|---|---|---|---|
| `mnist` | 28×28 grayscale digits | 10 | 60,000 | 10,000 |
| `cifar10` | 32×32 color objects | 10 | 50,000 | 10,000 |
| `imagenet32` | 32×32 color, ImageNet downsampled | 1000 | 1,281,167 | 50,000 (the official validation set) |

Every task uses the same format:

- **Images:** `X` is a `uint8` NumPy array with shape `(N, channels, height, width)` and values from 0 to 255.
- **Labels:** `y` is an `int64` array of class indices.

## The rules

**Allowed:**

- **Network pieces:** linear layers, activations, normalization layers, and residual connections, at any depth and width.
- **Training choices:** any loss, optimizer, learning-rate schedule, regularizer, or initialization.
- **Data handling:** augmentation, test-time augmentation, and patch-style tricks or mixing across positions, as long as you can explain them.

**Not allowed:**

- No convolutions, recurrent layers, or other new layer types.
- No ensembles.
- No pretrained weights and no distillation. The model must be trained from the data directly.
- No data beyond what each benchmark provides.
- No loading saved weights. `train()` must build and train the model from scratch.

**Required interface.** Your package must provide these two functions:

```python
model = train(task, X_train, y_train, seed=0, device="cuda")  # build and train from scratch
probs = predict(model, X_test)                                # (N, num_classes) probabilities
```

See the README in the baseline repository for the full assignment.

## 1. Setup

Edit these values, then run the notebook top to bottom.

In [ ]:
# The GitHub repository to test. Replace this with your own repository.
PACKAGE_URL = "git+https://github.com/<owner>/<repo>.git"

# The name you import the package by (the folder inside src/).
IMPORT_NAME = "mlp_baseline"

# Where the datasets are saved. Keep this folder between runs to skip downloading.
DATA_DIR = "data"

# Which tasks to run. Remove some while testing to save time.
TASKS_TO_RUN = ["mnist", "cifar10", "imagenet32"]

SEED = 0

In [ ]:
# Install the package straight from GitHub.
# --force-reinstall always grabs your latest commit, even if the version
# number did not change. --no-deps leaves the installed PyTorch (and its
# GPU support) alone.
%pip install -q --force-reinstall --no-deps {PACKAGE_URL}

# The Hugging Face `datasets` library downloads the benchmarks.
%pip install -q datasets

In [ ]:
import importlib
import os
import time

import numpy as np
import torch

student = importlib.import_module(IMPORT_NAME)   # your package
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Testing", IMPORT_NAME, "on", DEVICE)

## 2. Get the datasets

All three datasets come from the Hugging Face Hub. The first run downloads each one and saves it as a single `.npz` file in `DATA_DIR`. Later runs find that file and load it directly.

ImageNet 32×32 is the large one: about 0.7 GB to download and about 4 GB on disk once saved. Converting it the first time takes several minutes. ImageNet is released for non-commercial research and education only.

In [ ]:
# For each task: Hugging Face dataset name, image column, train split, test split.
SOURCES = {
    "mnist":      ("ylecun/mnist",                     "image", "train", "test"),
    "cifar10":    ("uoft-cs/cifar10",                  "img",   "train", "test"),
    "imagenet32": ("benjamin-paine/imagenet-1k-32x32", "image", "train", "validation"),
}
NUM_CLASSES = {"mnist": 10, "cifar10": 10, "imagenet32": 1000}


def to_numpy(split, image_column, mode):
    """Convert a Hugging Face split into uint8 arrays X (N, C, H, W) and y (N,)."""
    images, labels = [], []
    for start in range(0, len(split), 10_000):        # work in chunks of 10k images
        chunk = split[start:start + 10_000]
        # mode is "L" (grayscale) or "RGB"; convert() fixes any odd image formats.
        images.append(np.stack([np.asarray(img.convert(mode)) for img in chunk[image_column]]))
        labels.append(np.asarray(chunk["label"]))
        print(f"  converted {start + len(chunk['label']):,} / {len(split):,}", end="\r")
    print()
    X = np.concatenate(images)
    X = X[:, None] if mode == "L" else X.transpose(0, 3, 1, 2)   # -> (N, C, H, W)
    return np.ascontiguousarray(X), np.concatenate(labels).astype(np.int64)


def load_task(task):
    """Return X_train, y_train, X_test, y_test, downloading only if there is no saved copy."""
    path = os.path.join(DATA_DIR, f"{task}.npz")

    if os.path.exists(path):                           # saved copy found: load it
        data = np.load(path)
        return data["X_train"], data["y_train"], data["X_test"], data["y_test"]

    from datasets import load_dataset
    name, image_column, train_split, test_split = SOURCES[task]
    mode = "L" if task == "mnist" else "RGB"
    print(f"{task}: downloading {name} (first run only)")
    X_train, y_train = to_numpy(load_dataset(name, split=train_split), image_column, mode)
    X_test, y_test = to_numpy(load_dataset(name, split=test_split), image_column, mode)

    # Write to a temporary file, then rename it. An interrupted run then
    # never leaves a half-written file that looks like a finished one.
    os.makedirs(DATA_DIR, exist_ok=True)
    temp_path = path.replace(".npz", ".partial.npz")
    np.savez(temp_path, X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test)
    os.replace(temp_path, path)
    return X_train, y_train, X_test, y_test

In [ ]:
# Download (or load) every task once and check what we got.
for task in TASKS_TO_RUN:
    X_train, y_train, X_test, y_test = load_task(task)
    print(f"{task:>10}: X_train {X_train.shape} {X_train.dtype}, X_test {X_test.shape}, "
          f"labels {y_train.min()}..{y_train.max()}")
    del X_train, y_train, X_test, y_test             # free memory before the next one

In [ ]:
# A quick look at a few training images from each task.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(TASKS_TO_RUN), 8, figsize=(10, 1.4 * len(TASKS_TO_RUN)), squeeze=False)
for row, task in enumerate(TASKS_TO_RUN):
    X_train, y_train, _, _ = load_task(task)
    for col in range(8):
        img = X_train[col].transpose(1, 2, 0)          # (C, H, W) -> (H, W, C) for plotting
        axes[row, col].imshow(img.squeeze(), cmap="gray" if img.shape[2] == 1 else None)
        axes[row, col].set_title(f"{task}\nlabel {y_train[col]}" if col == 0 else f"label {y_train[col]}", fontsize=8)
        axes[row, col].axis("off")
    del X_train, y_train
plt.tight_layout()
plt.show()

## 3. Train and evaluate

For each task, this section:

1. Calls `train()` on the training set.
2. Calls `predict()` on the test set.
3. Checks the format of the output.
4. Computes test accuracy.

Accuracy is computed here, by the notebook, not by your code.

In [ ]:
results = []

for task in TASKS_TO_RUN:
    print(f"\n===== {task} =====")
    X_train, y_train, X_test, y_test = load_task(task)

    # Train from scratch.
    start = time.time()
    model = student.train(task, X_train, y_train, seed=SEED, device=DEVICE)
    train_minutes = (time.time() - start) / 60

    # Predict on the held-out test set.
    probs = torch.as_tensor(student.predict(model, X_test)).float().cpu()

    # Check that predict() returned one probability row per test image.
    assert probs.shape == (len(y_test), NUM_CLASSES[task]), f"predict() returned shape {tuple(probs.shape)}"
    assert torch.allclose(probs.sum(dim=1), torch.ones(len(probs)), atol=1e-3), "rows of predict() must sum to 1"

    accuracy = (probs.argmax(dim=1).numpy() == y_test).mean()
    results.append((task, accuracy, train_minutes))
    print(f"{task}: test accuracy {accuracy:.2%}  (training took {train_minutes:.1f} min)")

    del model, X_train, y_train, X_test, y_test, probs   # free memory before the next task
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 4. Results

In [ ]:
print(f"{'task':>10} | {'test accuracy':>13} | {'train time':>10}")
print("-" * 41)
for task, accuracy, minutes in results:
    print(f"{task:>10} | {accuracy:>13.2%} | {minutes:>6.1f} min")